# Pipeline de Datos con Patrón Medallón

**Volumen fuente:** `/Volumes/proyecto_bda/bda_schema/bda_volumen`  
**Archivos:** `reviews_raw.csv` y `BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv`

**Tablas Gold generadas:**
- `gold_reviews_final` — tabla maestra reseña + local + sentimiento
- `gold_stats_categoria` — estadísticas agregadas por categoría de restaurante
- `gold_features_usuario` — features por usuario para KMeans


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CATALOGO = "proyecto_bda"
SCHEMA   = "bda_schema"

#Rutas/ csv´s en el volumen
VOLUMEN_BASE = f"/Volumes/{CATALOGO}/{SCHEMA}/bda_volumen"
CSV_REVIEWS  = f"{VOLUMEN_BASE}/reviews_dataset.csv"
CSV_PLACES   = f"{VOLUMEN_BASE}/BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv"
#Tablas
TBL_BRONZE_REVIEWS    = f"{CATALOGO}.{SCHEMA}.bronze_reviews"
TBL_BRONZE_REVIEWS_STREAM  = f"{CATALOGO}.{SCHEMA}.bronze_reviews_stream" #Ultimo de Kafka
TBL_BRONZE_PLACES     = f"{CATALOGO}.{SCHEMA}.bronze_places"
TBL_SILVER_REVIEWS    = f"{CATALOGO}.{SCHEMA}.silver_reviews"
TBL_SILVER_PLACES     = f"{CATALOGO}.{SCHEMA}.silver_places"
# Gold: tres tablas con propósitos distintos
TBL_GOLD_FINAL        = f"{CATALOGO}.{SCHEMA}.gold_reviews_final"
TBL_GOLD_STATS_CAT    = f"{CATALOGO}.{SCHEMA}.gold_stats_categoria"
TBL_GOLD_FEAT_USUARIO = f"{CATALOGO}.{SCHEMA}.gold_features_usuario"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Configuración lista.")
print(f"  Fuente reviews : {CSV_REVIEWS}")
print(f"  Fuente locales : {CSV_PLACES}")
print(f"  Tablas destino : {CATALOGO}.{SCHEMA}.*")


Configuración lista.
  Fuente reviews : /Volumes/proyecto_bda/bda_schema/bda_volumen/reviews_dataset.csv
  Fuente locales : /Volumes/proyecto_bda/bda_schema/bda_volumen/BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv
  Tablas destino : proyecto_bda.bda_schema.*



## 1. Capa BRONZE - Ingesta Raw
Lee los CSVs del volumen y los guarda como tablas delta

In [0]:
# Bronze: Reseñas ───────────────────────────────────────────
def ejecutar_bronze_reviews():
    df_bronze = spark.read.format("csv").option("header", "true").option("sep", ",").option("encoding", "UTF-8").load(CSV_REVIEWS)
    print(f"Columnas: {df_bronze.columns}")
    # overwriteSchema=True permite reejecutar sin errores de esquema
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_BRONZE_REVIEWS)

    count = spark.table(TBL_BRONZE_REVIEWS).count()
    print(f"Registros en Bronze (Reseñas): {count:,}")
    print(f"Tabla guardada: {TBL_BRONZE_REVIEWS}\n")

ejecutar_bronze_reviews()

Columnas: ['place_id', 'id_review', 'caption', 'relative_date', 'review_date', 'retrieval_date', 'rating', 'username', 'n_review_user', 'n_photo_user', 'url_user', 'url_source']
Registros en Bronze (Reseñas): 3,431,251
Tabla guardada: proyecto_bda.bda_schema.bronze_reviews



In [0]:
#Bronze: Locales ───────────────────────────────────────────
def ejecutar_bronze_places():
    df_bronze = spark.read.format("csv").option("header", "true").option("sep", ",").option("quote", "\"").option("escape", "\"").option("encoding", "UTF-8").option("multiLine", "true").load(CSV_PLACES)

    print(f"Columnas: {df_bronze.columns}")
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_BRONZE_PLACES)

    count = spark.table(TBL_BRONZE_PLACES).count()
    print(f"Registros en Bronze (Locales): {count:,}")
    print(f"Tabla guardada: {TBL_BRONZE_PLACES}\n")

ejecutar_bronze_places()

Columnas: ['id', 'url_place', 'title', 'category', 'address', 'phoneNumber', 'completePhoneNumber', 'domain', 'url', 'coor', 'stars', 'reviews', 'source_query']
Registros en Bronze (Locales): 39,947
Tabla guardada: proyecto_bda.bda_schema.bronze_places



## 2. Capa SILVER - Limpieza y Transformación
Aplica la misma logica de limpieza que "BDA_entrega_parcial":
deduplicación, casteos, texto limpio, feature engineering temporal y de texto.

In [0]:
#Silver: Reseñas (batch + streaming unificados)
def ejecutar_silver_reviews():
    df_batch = spark.table(TBL_BRONZE_REVIEWS)
    try:
        df_stream = spark.table(TBL_BRONZE_REVIEWS_STREAM)
        n_stream = df_stream.count()
        print(f"  Streaming: {n_stream:,} registros adicionales")
        columnas = df_batch.columns
        df = df_batch.unionAll(df_stream.select(*columnas))
    except Exception:
        df = df_batch
        print("  Sin datos de streaming (bronze_reviews_stream no existe o está vacía)")

    #casteos seguros con try_cast (toleran filas corrompidas)
    df_silver=df.withColumn("rating",F.expr("try_cast(rating AS DOUBLE)")).withColumn("review_date",F.expr("try_cast(review_date AS TIMESTAMP)"))

    nulos_rating = df_silver.filter(F.col("rating").isNull()).count() 
    nulos_fecha  = df_silver.filter(F.col("review_date").isNull()).count()
    print(f"  Filas con rating inválido   : {nulos_rating:,}")
    print(f"  Filas con fecha inválida    : {nulos_fecha:,}")

    df_silver = df_silver.fillna({"caption": "", "username": "Usuario Anónimo"})
    df_silver = df_silver.dropDuplicates(["id_review"])
    df_silver = df_silver.filter(F.col("rating").isNotNull() & F.col("review_date").isNotNull())

    #  Feature engineering textual 
    df_silver = df_silver.withColumn("text_len",F.length(F.col("caption"))).withColumn("word_count", F.size(F.split(F.trim(F.col("caption")), r"\s+")))

    df_silver = df_silver.withColumn("caption_clean",F.regexp_replace(F.lower(F.col("caption")), r"[^a-záéíóúñ0-9\s]", ""))
    df_silver = df_silver.withColumn("rating_category",F.when(F.col("rating") <= 2, "Baja").when(F.col("rating") == 3,"Media").otherwise("Alta"))
    #separo la fecha en años,meses, dias
    df_silver = df_silver.withColumn("year",F.year("review_date")).withColumn("month",F.month("review_date")).withColumn("day_of_week", F.dayofweek("review_date"))

    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_SILVER_REVIEWS)

    count = spark.table(TBL_SILVER_REVIEWS).count()
    print(f"Registros limpios en Silver (Reseñas): {count:,}")
    print(f"Tabla guardada: {TBL_SILVER_REVIEWS}\n")

ejecutar_silver_reviews()

  Streaming: 0 registros adicionales
  Filas con rating inválido   : 18,658
  Filas con fecha inválida    : 5,073
Registros limpios en Silver (Reseñas): 2,764,316
Tabla guardada: proyecto_bda.bda_schema.silver_reviews



In [0]:
#Silver: Locales 
def ejecutar_silver_places():
    df = spark.table(TBL_BRONZE_PLACES)
    #  Selección y renombrado de columnas 
    #    title -> name así se usa en BDA_entrega_parcial
    df_silver = df.select(F.col("id"),
        F.col("title").alias("name"),
        F.col("category"),
        F.col("address"),
        F.col("phoneNumber"),
        F.col("coor"),
        F.col("stars").cast(DoubleType()).alias("avg_rating"),
        F.when(F.col("url") == "", None).otherwise(F.col("url")).alias("url"),
        F.col("url_place")
    )

    # Extracción de lat/lng desde columna 'coor' ('lat, lng')
    #  Equivalente a df_places['coor'].str.split(',') del BDA
    df_silver = df_silver.withColumn("latitude",F.trim(F.split(F.col("coor"), ",").getItem(0)).cast(DoubleType())) \
        .withColumn("longitude",F.trim(F.split(F.col("coor"), ",").getItem(1)).cast(DoubleType()))

    #Deduplicación por id del local 
    df_silver = df_silver.dropDuplicates(["id"])
    # Relleno de nombre nulo 
    df_silver = df_silver.fillna({"name": "Restaurante Desconocido"})
    #  Descartamos locales sin coordenadas válidas
    df_silver = df_silver.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())

    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_SILVER_PLACES)

    count = spark.table(TBL_SILVER_PLACES).count()
    print(f"Registros limpios en Silver (Locales): {count:,}")
    print(f"Tabla guardada: {TBL_SILVER_PLACES}\n")

ejecutar_silver_places()

Registros limpios en Silver (Locales): 39,945
Tabla guardada: proyecto_bda.bda_schema.silver_places



## 3. Capa GOLD - Agregaciones y Features para Modelos
>`gold_reviews_final` - tabla maestra (join reseñas + locales + sentimiento)
>`gold_stats_categoria` - estadísticas agregadas por categoría (para análisis descriptivo)
>`gold_features_usuario` - features por usuario listos para KMeans


In [0]:
# Gold: tabla maestra (join + sentimiento)
def ejecutar_gold_maestra():
    print("Tabla maestra")
    df_reviews = spark.table(TBL_SILVER_REVIEWS)
    df_places  = spark.table(TBL_SILVER_PLACES)

    df_gold = df_reviews.join(df_places,df_reviews.place_id == df_places.id,"left").drop(df_places.id)

    df_gold = df_gold.withColumn("sentimiento",
        F.when(F.col("rating") >= 4,"Positivo").when(F.col("rating") == 3,"Neutral").otherwise("Negativo"))

    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_FINAL)

    count = spark.table(TBL_GOLD_FINAL).count()
    print(f"  Registros: {count:,}")
    print(f"  Tabla: {TBL_GOLD_FINAL}\n")

    spark.table(TBL_GOLD_FINAL).createOrReplaceTempView("tabla_maestra_reviews")
    print("Vista temporal de: tabla_maestra_reviews registrada.")

ejecutar_gold_maestra()

Tabla maestra
  Registros: 2,764,316
  Tabla: proyecto_bda.bda_schema.gold_reviews_final

Vista temporal de: tabla_maestra_reviews registrada.


In [0]:
# Gold: estadísticas agregadas por categoría
def ejecutar_gold_stats_categoria():
    print("Estadísticas por categoría")
    df = spark.table(TBL_GOLD_FINAL)

    df_stats = df.groupBy("category").agg(
        F.count("id_review").alias("total_resenas"),
        F.round(F.avg("rating"), 2).alias("rating_promedio"),
        F.round(F.stddev("rating"), 2).alias("rating_desviacion"),
        F.sum(F.when(F.col("sentimiento") == "Positivo", 1).otherwise(0)).alias("resenas_positivas"),
        F.sum(F.when(F.col("sentimiento") == "Negativo", 1).otherwise(0)).alias("resenas_negativas"),
        F.round(F.avg("word_count"), 1).alias("promedio_palabras"),
        F.countDistinct("place_id").alias("total_locales")
    ).withColumn(
        "porcentaje_aprobacion",
        F.round(F.col("resenas_positivas") / F.col("total_resenas") * 100, 2)
    ).filter(F.col("category").isNotNull()) \
     .orderBy(F.desc("total_resenas"))

    df_stats.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_STATS_CAT)

    count = spark.table(TBL_GOLD_STATS_CAT).count()
    print(f"  Categorías procesadas: {count:,}")
    print(f"  Tabla: {TBL_GOLD_STATS_CAT}\n")
    display(spark.table(TBL_GOLD_STATS_CAT).limit(10))

ejecutar_gold_stats_categoria()

Estadísticas por categoría
  Categorías procesadas: 128
  Tabla: proyecto_bda.bda_schema.gold_stats_categoria



category,total_resenas,rating_promedio,rating_desviacion,resenas_positivas,resenas_negativas,promedio_palabras,total_locales,porcentaje_aprobacion
Restaurante,1411524,4.12,1.21,1086550,158861,7.7,17499,76.98
Restaurante peruano,201185,4.25,1.15,162725,19453,9.7,1629,80.88
Pizzería,161541,4.24,1.15,130244,15352,8.0,2095,80.63
Restaurante chino,108668,3.98,1.25,79145,14508,7.2,1020,72.83
Cafetería,102155,4.31,1.1,84626,8559,9.9,1424,82.84
Restaurante de comida rápida,98918,4.09,1.25,75280,12328,7.0,2132,76.1
Restaurante especializado en pollo,96614,4.03,1.27,72018,12721,7.2,1316,74.54
Panadería,78397,4.22,1.12,62441,6861,6.9,2303,79.65
Heladería,67229,4.37,1.06,56712,4985,8.1,1852,84.36
Marisquería,54478,4.11,1.2,41683,6196,7.2,797,76.51


In [0]:
#Gold: features por usuario para KMeans despues
# Estas features son las que consume BDA_entrega_parcial
# para el clustering, reemplazando el groupby de pandas
def ejecutar_gold_features_usuario():
    print("Gold 3/3: Features por usuario (para KMeans)")
    df = spark.table(TBL_GOLD_FINAL)

    df_feat = df.groupBy("username").agg(
        F.count("id_review").alias("review_count"),
        F.round(F.avg("rating"), 4).alias("avg_rating"),
        F.round(F.stddev("rating"), 4).alias("std_rating"),
        F.round(F.avg("word_count"), 4).alias("avg_word_count"),
        # Transformación log para corregir asimetría (igual que en BDA pandas)
        F.round(F.log1p(F.count("id_review")), 4).alias("log_review_count"),
        F.round(F.log1p(F.avg("word_count")), 4).alias("log_avg_word_count")
    ).fillna({"std_rating": 0.0})  # usuarios con 1 sola reseña tienen std=null

    df_feat.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_FEAT_USUARIO)

    count = spark.table(TBL_GOLD_FEAT_USUARIO).count()
    print(f"  Usuarios procesados: {count:,}")
    print(f"  Tabla: {TBL_GOLD_FEAT_USUARIO}\n")
    display(spark.table(TBL_GOLD_FEAT_USUARIO).limit(5))

ejecutar_gold_features_usuario()

Gold 3/3: Features por usuario (para KMeans)
  Usuarios procesados: 1,200,383
  Tabla: proyecto_bda.bda_schema.gold_features_usuario



username,review_count,avg_rating,std_rating,avg_word_count,log_review_count,log_avg_word_count
angel leandro bernabe,1,5.0,0.0,2.0,0.6931,1.0986
David Klauber,1,5.0,0.0,7.0,0.6931,2.0794
Pablo Arevalo Moran,1,4.0,0.0,4.0,0.6931,1.6094
Carlos Alberto Villarreal Leon,5,3.4,0.5477,1.0,1.7918,0.6931
Matías,11,4.3636,0.809,8.3636,2.4849,2.2368


## 4 Validación del Pipeline

In [0]:
tablas = {
    "Bronze Reviews":      TBL_BRONZE_REVIEWS,
    "Bronze Places":       TBL_BRONZE_PLACES,
    "Silver Reviews":      TBL_SILVER_REVIEWS,
    "Silver Places":       TBL_SILVER_PLACES,
    "Gold Maestra":        TBL_GOLD_FINAL,
    "Gold Stats Categoria": TBL_GOLD_STATS_CAT,
    "Gold Feat Usuario":   TBL_GOLD_FEAT_USUARIO,
}

print(f"{'Tabla':<25} {'Registros':>12} {'Estado':>8}")
print("=" * 50)
for nombre, tabla in tablas.items():
    try:
        n = spark.table(tabla).count()
        print(f"{nombre:<25} {n:>12,} {'OK':>8}")
    except Exception as e:
        print(f"{nombre:<25} {'—':>12} {'ERROR':>8}: {e}")


Tabla                        Registros   Estado
Bronze Reviews               3,431,251       OK
Bronze Places                   39,947       OK
Silver Reviews               2,764,316       OK
Silver Places                   39,945       OK
Gold Maestra                 2,764,316       OK
Gold Stats Categoria               128       OK
Gold Feat Usuario            1,200,383       OK


In [0]:
# Vista previa de la tabla Gold
display(spark.table(TBL_GOLD_FINAL).limit(10))

place_id,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,n_photo_user,url_user,url_source,text_len,word_count,caption_clean,rating_category,year,month,day_of_week,name,category,address,phoneNumber,coor,avg_rating,url,url_place,latitude,longitude,sentimiento
ChIJx4RKUu7JBZERgPHxYOmm3rM,ChZDSUhNMG9nS0VJQ0FnSUNlMDRyUldnEAE,Una de las mejores mieles de picarones,Hace 3 años,2023-04-27T08:00:43.352Z,2026-04-26 08:00:43.352357,4.0,Cesar Peredo,244.0,null,https://www.google.com/maps/contrib/114526674593988986467/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJx4RKUu7JBZERgPHxYOmm3rM,38,7,una de las mejores mieles de picarones,Alta,2023,4,5,El puntito dulce,Cafetería,"Jr. Huiracocha 1356, Jesús María 15072",null,"-12.0757256,-77.0479639",4.1,https://m.facebook.com/ElPuntitoDulcePeru/,https://www.google.com/maps/place/?q=place_id:ChIJx4RKUu7JBZERgPHxYOmm3rM,-12.0757256,-77.0479639,Positivo
ChIJSyhlZADJBZERyGu47s9EHB0,Ci9DQUlRQUNvZENodHljRjlvT25WbkxTMXNWRTFKY1c5RVZ6RmFYMlZaWjJaaGFYYxAB,"Me encantó! , la comida y el servicio fueron A1, definitivamente volveré!",Hace 6 meses,2025-10-28T08:00:45.397Z,2026-04-26 08:00:45.397604,5.0,marita uceda paucar,0.0,null,https://www.google.com/maps/contrib/117224966306903836469/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJSyhlZADJBZERyGu47s9EHB0,73,12,me encantó la comida y el servicio fueron a1 definitivamente volveré,Alta,2025,10,3,HANA SUSHI & NIKKEI CUISINE,Restaurante japonés,"Av. Ignacio Merino 1765, Lince 15046",982 368 432,"-12.081723199999999,-77.0327265",4.6,https://www.instagram.com/hanasushinikkeicuisine?igsh=MXIxNHZwcHF3cGxvcg==,https://www.google.com/maps/place/?q=place_id:ChIJSyhlZADJBZERyGu47s9EHB0,-12.081723199999999,-77.0327265,Positivo
ChIJLe8wKDjPBZER9i99XQ6O3Mo,ChdDSUhNMG9nS0VJQ0FnTUNZekppb2hRRRAB,Muy rico.....lo recomiendo,Hace 11 meses,2025-05-31T08:01:42.541Z,2026-04-26 08:01:42.541632,5.0,Heber Fernández Aguilar,3.0,null,https://www.google.com/maps/contrib/114612614686506524153/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJLe8wKDjPBZER9i99XQ6O3Mo,26,3,muy ricolo recomiendo,Alta,2025,5,7,Corralito,Restaurante especializado en pollo,"Av. Felipe Arancibia 537, Rímac 15094",(01) 4136180,"-12.0295404,-77.0356554",4.2,null,https://www.google.com/maps/place/?q=place_id:ChIJLe8wKDjPBZER9i99XQ6O3Mo,-12.0295404,-77.0356554,Positivo
ChIJ3Q6EpjzNBZERt4mFDGFzclA,Ci9DQUlRQUNvZENodHljRjlvT2xoVFRYVkRVMlZFZUhwQ1ZUQmZWazFSYmxkUlFtYxAB,Mala experiencia,Hace 3 semanas,2026-04-05T08:01:52.971Z,2026-04-26 08:01:52.971397,1.0,Marcelo Márquez,0.0,null,https://www.google.com/maps/contrib/109419766876397833202/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJ3Q6EpjzNBZERt4mFDGFzclA,16,2,mala experiencia,Baja,2026,4,1,Perusuyo,Restaurante peruano,"XV9M+MJH, Callao 15057",null,"-12.0303556,-77.11612699999999",4.1,https://www.perusuyo.pe/,https://www.google.com/maps/place/?q=place_id:ChIJ3Q6EpjzNBZERt4mFDGFzclA,-12.0303556,-77.11612699999999,Negativo
ChIJLe8wKDjPBZER9i99XQ6O3Mo,ChdDSUhNMG9nS0VJQ0FnSURPaGRHSTZnRRAB,Todo riquísimo 😋…,Hace 3 años,2023-04-27T08:02:05.768Z,2026-04-26 08:02:05.768583,5.0,claudia mendoza,23.0,null,https://www.google.com/maps/contrib/112787605636602557285/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJLe8wKDjPBZER9i99XQ6O3Mo,17,3,todo riquísimo,Alta,2023,4,5,Corralito,Restaurante especializado en pollo,"Av. Felipe Arancibia 537, Rímac 15094",(01) 4136180,"-12.0295404,-77.0356554",4.2,null,https://www.google.com/maps/place/?q=place_id:ChIJLe8wKDjPBZER9i99XQ6O3Mo,-12.0295404,-77.0356554,Positivo
ChIJ3Q6EpjzNBZERt4mFDGFzclA,Ci9DQUlRQUNvZENodHljRjlvT2xoZldUVTNXVnBrTldaWVMycDBTV0YwUW5SdWJtYxAB,"Muy buena nuestra experiencia aquí, Kaelly es muy calida y fue muy buena anfitriona con sus recomendaciones, el mejor chicharrón que hemos comido ñ, regresaremos cuando regresemos a Chile, no hay pierde.",Hace un mes,2026-03-27T08:02:19.982Z,2026-04-26 08:02:19.982814,5.

## 5 Insights SQL sobre la Capa Gold

In [0]:
# Recarga de vista por si se reinicia el notebook
spark.table(TBL_GOLD_FINAL).createOrReplaceTempView("tabla_maestra_reviews")
print("Vista 'tabla_maestra_reviews' registrada")

Vista 'tabla_maestra_reviews' registrada


In [0]:
# Insight 1: Ranking de Satisfacción por Categoría 
spark.sql("""
    SELECT
        category                                                          AS Categoria,
        COUNT(id_review)                                                  AS Total_Resenas,
        ROUND(AVG(rating), 2)                                             AS Calificacion_Promedio,
        SUM(CASE WHEN sentimiento = 'Positivo' THEN 1 ELSE 0 END)        AS Resenas_Positivas,
        SUM(CASE WHEN sentimiento = 'Negativo' THEN 1 ELSE 0 END)        AS Quejas,
        ROUND(
            SUM(CASE WHEN sentimiento = 'Positivo' THEN 1 ELSE 0 END)
            / COUNT(id_review) * 100, 2
        )                                                                 AS Porcentaje_Aprobacion
    FROM tabla_maestra_reviews
    WHERE category IS NOT NULL
    GROUP BY category
    HAVING Total_Resenas > 500
    ORDER BY Porcentaje_Aprobacion DESC
    LIMIT 8
""").show(truncate=False)

+------------------------------+-------------+---------------------+-----------------+------+---------------------+
|Categoria                     |Total_Resenas|Calificacion_Promedio|Resenas_Positivas|Quejas|Porcentaje_Aprobacion|
+------------------------------+-------------+---------------------+-----------------+------+---------------------+
|Restaurante orgánico          |1673         |4.73                 |1566             |54    |93.6                 |
|Restaurante francés           |698          |4.64                 |641              |29    |91.83                |
|Chocolatería                  |1616         |4.56                 |1450             |71    |89.73                |
|Restaurante de cocina española|933          |4.52                 |830              |45    |88.96                |
|Restaurante vegano            |10757        |4.56                 |9567             |647   |88.94                |
|Cervecería artesanal          |1932         |4.49                 |1717

In [0]:
# Insight 2: Tracción de Grandes Cadenas
spark.sql("""
    SELECT
        name                          AS Nombre_Restaurante,
        COUNT(DISTINCT place_id)      AS Sucursales_Registradas,
        COUNT(id_review)              AS Total_Interacciones,
        ROUND(AVG(rating), 2)         AS Calificacion_Global,
        MAX(category)                 AS Categoria_Principal
    FROM tabla_maestra_reviews
    WHERE name != 'Restaurante Desconocido'
      AND name IS NOT NULL
    GROUP BY name
    HAVING Total_Interacciones > 1000
    ORDER BY Total_Interacciones DESC
    LIMIT 20
""").show(truncate=False)

+------------------------------+----------------------+-------------------+-------------------+----------------------------------+
|Nombre_Restaurante            |Sucursales_Registradas|Total_Interacciones|Calificacion_Global|Categoria_Principal               |
+------------------------------+----------------------+-------------------+-------------------+----------------------------------+
|KFC                           |85                    |17742              |3.88               |Restaurante especializado en pollo|
|Norky's                       |21                    |5088               |3.93               |Restaurante especializado en pollo|
|Roky's                        |18                    |4788               |3.86               |Restaurante peruano               |
|Bembos                        |22                    |4399               |3.96               |Restaurante de comida rápida      |
|McDonald's                    |19                    |4113               |3.86    

In [0]:
# Insight 3: Engagement por Categoría de Rating
spark.sql("""
    SELECT
        rating_category            AS Categoria_Rating,
        COUNT(id_review)           AS Total_Reviews,
        ROUND(AVG(text_len), 1)    AS Promedio_Chars,
        ROUND(AVG(word_count), 1)  AS Promedio_Palabras
    FROM tabla_maestra_reviews
    GROUP BY rating_category
    ORDER BY Total_Reviews DESC
""").show()

+----------------+-------------+--------------+-----------------+
|Categoria_Rating|Total_Reviews|Promedio_Chars|Promedio_Palabras|
+----------------+-------------+--------------+-----------------+
|            Alta|      2162842|          38.8|              7.0|
|           Media|       303666|          39.9|              7.6|
|            Baja|       297808|          89.3|             16.4|
+----------------+-------------+--------------+-----------------+

